# Time Series Analysis and Forecasting
## A classroom-friendly walkthrough using the AirPassengers dataset

**Learning goal:** Understand how time-dependent data behaves and build models that forecast future airline passenger numbers.

### What we will learn
1. What time series data is and why time order matters
2. How to load, inspect, and visualize time series data
3. Trend, seasonality, noise, and stationarity
4. How ACF and PACF help us understand past relationships
5. How log transformation and differencing work
6. How to build a simple baseline, AR, ARIMA, and SARIMA models
7. How to evaluate forecasts and predict future passenger numbers

> **Dataset note:** Keep `AirPassengers.csv` in the same folder as this notebook. It should contain a `Month` column and a `#Passengers` column. The dataset contains monthly passenger totals from January 1949 to December 1960.

## 1. What is time series data?

A **time series** is a set of observations recorded in time order, usually at regular intervals.

Examples:
- Daily temperature
- Monthly sales
- Hourly electricity usage
- Daily stock prices
- Monthly airline passengers

### Why does time order matter?

In ordinary tabular data, rows are often treated as independent observations. In time series data, today's value may depend on yesterday's or last month's value.

For example, passenger traffic in December may be related to passenger traffic in the previous December. We must preserve the time order when preparing training and testing data.

### Our problem

We have historical monthly passenger counts. We want to learn patterns from the past and predict passenger counts for months we have not observed yet.

## 2. Import the libraries

A library is a collection of ready-made tools. We import only the tools needed for this analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

### Why are these libraries used?

- **pandas:** loads and organizes the monthly data.
- **NumPy:** performs numerical calculations such as logarithms.
- **Matplotlib:** draws time series plots.
- **statsmodels:** provides time series tests and forecasting models.
- **scikit-learn:** provides common error metrics.
- **warnings:** hides non-critical warning messages so the teaching output is easier to read. During debugging, warnings can be turned back on.

## 3. Load the dataset

We use `read_csv()` to read the CSV file into a pandas DataFrame.

A DataFrame is a table with rows and columns.

In [ ]:
data = pd.read_csv("AirPassengers.csv")
data.head()

**What should we notice?**

- `Month` contains the month for each observation.
- `#Passengers` contains the passenger count.
- Each row represents one month.

`head()` displays the first five rows. It is a quick way to check whether the file loaded correctly.

### Inspect the last rows and the size of the dataset

`tail()` displays the last five rows. `shape` tells us the number of rows and columns.

In [ ]:
print("Dataset shape:", data.shape)
data.tail()

### Check column types and missing values

`info()` shows column names, data types, and non-null counts. If a column has fewer non-null values than the total number of rows, it may contain missing values.

In [ ]:
data.info()

## 4. Convert the month column into a proper date

A date stored as text is not as useful for time series operations as a real datetime value.

We convert `Month` to datetime and use it as the index. The **index** identifies each row; here, it identifies the month of the passenger count.

In [ ]:
data["Month"] = pd.to_datetime(data["Month"])
data = data.set_index("Month")

data.head()

### Rename the passenger column

The original column name `#Passengers` is awkward to type. We rename it to `Passengers`.

`inplace=True` changes the existing DataFrame. We then select the passenger column to work with a single time series.

In [ ]:
data = data.rename(columns={"#Passengers": "Passengers"})
passengers = data["Passengers"]

print("Start:", passengers.index.min())
print("End:", passengers.index.max())
print("Number of observations:", len(passengers))

### Tell pandas that observations are monthly

The data is recorded at the start of each month. Setting the frequency to `MS` means **Month Start**.

A regular date frequency helps time series tools understand the spacing between observations.

In [ ]:
passengers = passengers.asfreq("MS")
data = passengers.to_frame()
data.head()

## 5. Basic statistical understanding

Before building a model, understand the values.

- **Mean:** average passenger count.
- **Standard deviation:** how spread out the values are around the mean.
- **Minimum:** smallest observed value.
- **Maximum:** largest observed value.
- **Quartiles:** values that divide the observations into four parts.

`describe()` calculates these summary statistics.

In [ ]:
passengers.describe()

## 6. Visualize the time series

A line plot places time on the horizontal axis and passenger count on the vertical axis.

We use a line plot because the order and movement of values over time are important.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(passengers)
plt.title("Monthly AirPassengers (1949–1960)")
plt.xlabel("Month")
plt.ylabel("Number of passengers")
plt.grid(True)
plt.show()

### Read the graph

Look for three things:

1. **Trend:** the long-term direction. Is the series generally increasing or decreasing?
2. **Seasonality:** a pattern that repeats at a fixed interval. Do similar rises and falls occur each year?
3. **Noise:** short-term movements that are not explained by the trend or seasonal pattern.

In this dataset, passenger numbers generally increase over time, and a repeating yearly pattern is visible. The size of the seasonal swings also becomes larger as the series grows.

## 7. Decompose the time series

Decomposition separates a time series into components.

A common additive representation is:

\[
Observed = Trend + Seasonal + Residual
\]

- **Observed:** the original data.
- **Trend:** the long-term movement.
- **Seasonal:** the repeating pattern.
- **Residual:** what remains after removing trend and seasonality.

We use a period of 12 because the observations are monthly and we expect a yearly seasonal pattern.

In [ ]:
decomposition = seasonal_decompose(passengers, model="additive", period=12)
decomposition.plot()
plt.tight_layout()
plt.show()

**How to explain the output:** The trend component shows the long-term direction. The seasonal component shows the typical within-year pattern. The residual component contains what the trend and seasonal components do not explain.

The additive decomposition is a descriptive tool. It does not automatically make the data stationary.

## 8. What is stationarity?

A time series is **stationary** when its statistical behaviour stays broadly stable over time.

For a weakly stationary series, we usually discuss:

1. **Constant mean:** the average level does not systematically change over time.
2. **Constant variance:** the size of fluctuations remains broadly similar.
3. **Stable autocovariance:** the relationship between two observations depends on the time gap between them, not on the particular date.

### Why does stationarity matter?

Some classical time series methods work best when the series being modelled is stationary. Trend and changing seasonal behaviour can make patterns harder for those models to learn.

The original AirPassengers series has a clear upward trend and changing seasonal amplitude, so we investigate transformations and differencing.

## 9. Autocorrelation: are past values related to current values?

**Autocorrelation** measures the relationship between a time series and a shifted version of itself.

A **lag** is the number of time steps by which we shift the series.

- Lag 1: compare each month with the previous month.
- Lag 12: compare each month with the same month one year earlier.

The ACF (Autocorrelation Function) plot displays autocorrelation at different lags.

### Before running this code: What is ACF and why do we need it?

**ACF** means *Autocorrelation Function*. It measures how strongly a time series is related to its own earlier values.

A **lag** means how many time steps back we compare:
- Lag 1: this month compared with the previous month.
- Lag 2: this month compared with two months ago.
- Lag 12: this month compared with the same month last year.

`lags=36` asks the graph to show up to 36 months (three years) of past relationships. It does not remove rows or limit the dataset to 36 observations.

**Axes in the ACF graph:**
- X-axis: lag number (months back).
- Y-axis: autocorrelation value, usually between -1 and +1. Near +1 means a strong positive relationship; near -1 means a strong negative relationship; near 0 means a weak linear relationship.

We need ACF because forecasting models can use information from past observations. ACF helps us see whether past values are related and whether yearly seasonality may exist.

In [ ]:
plot_acf(passengers, lags=36)
plt.title("ACF of the original passenger series")
plt.show()

### How to read an ACF plot

- **Horizontal (X) axis:** lag number. Lag 1 means one month back; lag 12 means twelve months back.
- **Vertical (Y) axis:** autocorrelation value. Near +1 means strong positive relationship, near -1 means strong negative relationship, and near 0 means weak linear relationship.
- **Shaded region:** an approximate confidence region around zero.
- **Bars outside the shaded region:** may indicate autocorrelation that is statistically meaningful.

**Example:** A high bar at lag 12 suggests that passenger counts are related to the counts 12 months earlier. This can be a clue of yearly seasonality in monthly data.

A slowly declining ACF can be a clue of trend or non-stationary behaviour. Repeating peaks around lags 12, 24, and 36 can indicate yearly seasonality.

**Why do we need this plot?** It helps us understand whether past values contain useful information and gives clues about trend and seasonality. ACF is a diagnostic clue, not a complete stationarity test.

## 10. Test stationarity using the Augmented Dickey-Fuller test

The **ADF test** is a statistical test commonly used to investigate whether a series has a unit root.

Its hypotheses are:

- **Null hypothesis (H₀):** the series has a unit root (evidence of non-stationarity).
- **Alternative hypothesis (H₁):** the series does not have a unit root (evidence in favour of stationarity under the test assumptions).

A common teaching threshold is 0.05:
- p-value below 0.05: reject the null hypothesis.
- p-value at or above 0.05: do not reject the null hypothesis.

Do not say that a p-value proves stationarity. It is evidence from one test and should be considered with plots and domain knowledge.

### Before running this code: What is the ADF test?

The **Augmented Dickey-Fuller (ADF) test** is a statistical test used to investigate whether a time series has a unit root, which is commonly associated with non-stationarity.

The hypotheses are:
- **H₀ (null hypothesis):** the series has a unit root; this is evidence consistent with non-stationarity.
- **H₁ (alternative hypothesis):** the series does not have a unit root; this is evidence in favour of stationarity under the test assumptions.

We need this test because a graph and ACF plot give visual clues, while ADF provides a numerical test and a p-value.

The result is returned as a collection:
- `adf_result[0]`: ADF test statistic.
- `adf_result[1]`: p-value.
- `adf_result[2]`: number of lags used.
- `adf_result[3]`: number of observations used.

Python uses zero-based indexing: `[0]` means the first item, `[1]` means the second item, and so on.

In [ ]:
adf_result = adfuller(passengers)

print("ADF statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Number of lags used:", adf_result[2])
print("Number of observations:", adf_result[3])

### How do we make a decision from the ADF test?

We compare the **p-value** with **0.05** (5%):
- If p-value < 0.05: reject H₀. We have evidence against a unit root.
- If p-value >= 0.05: do not reject H₀. We do not have enough evidence to rule out a unit root.

**Important:** "Do not reject H₀" does not prove that the series is non-stationary. It means this test did not provide strong enough evidence against the null hypothesis.

Use the ADF result together with the time-series graph and ACF plot, rather than relying on only one test.

### Understand the ADF decision code

`adf_result` contains the values returned by the ADF test. `adf_result[1]` selects the second item, which is the p-value. We store it in the clearly named variable `p_value`.

We compare it with **0.05**, a commonly used significance level:
- p-value < 0.05: reject H₀; evidence against a unit root.
- p-value >= 0.05: do not reject H₀; insufficient evidence against a unit root.

`if` checks the condition. If it is true, Python runs the first `print()`. Otherwise, `else` runs the second `print()`.

**Important:** Do not reject H₀ does not prove non-stationarity. It means this test did not provide enough evidence to reject the null hypothesis.

In [ ]:
p_value = adf_result[1]

if p_value < 0.05:
    print("Reject H0: evidence against a unit root.")
else:
    print("Do not reject H0: insufficient evidence against a unit root.")

## 11. Log transformation: reduce changing variation

In the original graph, seasonal movements become larger as passenger numbers increase.

A **log transformation** compresses large values more than small values. This can help stabilize changing variance and make multiplicative patterns easier to model.

We use the natural logarithm, written mathematically as ln and calculated in Python using `np.log()`. Passenger counts are positive, so the logarithm is defined.

The log transformation changes the scale; it does not necessarily remove trend or seasonality.

### Before running this code: What is a log transformation?

A logarithm changes the scale of a number. `np.log()` calculates the natural logarithm (ln).

Large values are compressed more than small values. For example, the difference between 100 and 200 becomes smaller on the log scale than the difference on the original scale.

**Why do we need it here?** In our original graph, seasonal ups and downs become larger as the passenger count grows. A log transformation can help reduce this changing variation and make multiplicative growth easier to model.

Line by line:
- `np.log(passengers)` calculates the natural logarithm of every passenger count.
- `log_passengers = ...` stores the transformed time series.
- `plt.plot(log_passengers)` draws the transformed values over time.
- The title and axis labels explain what is being displayed.

A log transformation does not automatically remove the upward trend or seasonality. We check the result again.

In [ ]:
log_passengers = np.log(passengers)

plt.figure(figsize=(12, 5))
plt.plot(log_passengers)
plt.title("Log-transformed passenger series")
plt.xlabel("Month")
plt.ylabel("Log of passengers")
plt.grid(True)
plt.show()

### Compare stationarity after the log transformation

Look at the transformed graph and compare it with the original passenger graph.

Ask: **Have the fluctuations become more consistent in size?**

A log transformation can help stabilize variance, but it does not necessarily remove the trend. That is why we run the ADF test again in the next cell.

Remember:
- Log transformation changes the scale of values.
- Differencing calculates changes between time points.
- They solve different problems and can be used together when appropriate.

### Why run the ADF test again after taking logs?

We first tested the original passenger counts. Now we test the log-transformed values.

This checks whether changing the scale has also changed the evidence about stationarity:
- `adfuller(log_passengers)` runs the ADF test on the log series.
- `log_adf[0]` is the ADF statistic.
- `log_adf[1]` is the p-value.

Compare this p-value with the original series' p-value. If it is still 0.05 or higher, we do not have enough evidence to reject the unit-root null hypothesis. The log transformation may help with changing variance, but trend may remain. Differencing may still be needed.

In [ ]:
log_adf = adfuller(log_passengers)
print("ADF statistic:", log_adf[0])
print("p-value:", log_adf[1])

## 12. Differencing: focus on change rather than level

Differencing subtracts an earlier observation from the current observation.

First difference:

\[
Y'_t = Y_t - Y_{t-1}
\]

In plain language: **this month's value minus last month's value**.

Differencing can remove a changing level or trend. It also creates one missing value at the beginning because there is no previous observation for the first month.

### What do Y and t mean in the differencing formula?

In \(Y'_t = Y_t - Y_{t-1}\):
- `Y` represents the value being measured (here, passenger count or log passenger count).
- `t` represents the current time position (the current month).
- `t-1` represents the previous time position (the previous month).
- `Y_t` means the value in the current month.
- `Y_{t-1}` means the value in the previous month.
- The prime mark in `Y'` means we created a transformed/differenced series.

Example: if this month's value is 500 and last month's value is 470, the difference is 500 - 470 = 30. It tells us how much the value changed.

**Do we need a Python variable named `Y`?** No. `Y` is mathematical notation. In our code, the variables are named `passengers` and `log_passengers`.


### What does `.diff()` do, and why do we need it?

Differencing subtracts the previous time step from the current time step.

For each month it calculates:

**Current month's log value − previous month's log value**

`log_passengers.diff()` performs this calculation for every row. This is called **first differencing** because each value is compared with one previous time step.

The first row becomes `NaN` because January 1949 has no previous month in this dataset. `.dropna()` removes that one unavailable difference.

We use differencing to remove or reduce a changing level/trend. It may help make the series stationary, but we must check the result using a plot and the ADF test.

In [ ]:
first_difference = log_passengers.diff()

first_difference = first_difference.dropna()

plt.figure(figsize=(12, 4))
plt.plot(first_difference)
plt.title("First difference of log passengers")
plt.xlabel("Month")
plt.ylabel("Change in log passengers")
plt.grid(True)
plt.show()

### Test the differenced series

We run ADF again. Compare its p-value with the result for the original and log-transformed series.

In [ ]:
diff_adf = adfuller(first_difference)
print("ADF statistic:", diff_adf[0])
print("p-value:", diff_adf[1])

### Seasonal differencing

For monthly data with a yearly pattern, seasonal differencing compares a month with the same month in the previous year:

\[
Y'_t = Y_t - Y_{t-12}
\]

This specifically targets a repeating 12-month pattern. Seasonal differencing and ordinary first differencing solve related but different problems. We do not keep differencing automatically: excessive differencing can remove useful structure and make modelling worse.

In [ ]:
seasonal_difference = log_passengers.diff(12).dropna()

plt.figure(figsize=(12, 4))
plt.plot(seasonal_difference)
plt.title("Seasonal difference of log passengers (lag 12)")
plt.xlabel("Month")
plt.ylabel("Seasonal change in log passengers")
plt.grid(True)
plt.show()

## 13. ACF and PACF after differencing

The **PACF (Partial Autocorrelation Function)** measures the relationship at a particular lag after accounting for the shorter lags in between.

A simple classroom distinction:
- ACF: overall relationship with earlier observations.
- PACF: direct relationship with a particular earlier lag after accounting for intermediate lags.

ACF and PACF plots can help suggest AR and MA terms. They are guides; model selection should also consider diagnostics and test-set performance.

### Before running this code: Why do we plot both ACF and PACF?

We have created `first_difference`, the month-to-month change in the log passenger series. Now we inspect its relationships with past values.

- **ACF:** shows the overall relationship with earlier observations, including indirect relationships.
- **PACF:** shows the relationship with a particular lag after accounting for the shorter lags in between.

We use both plots as clues when choosing ARIMA model terms. They do not automatically tell us the perfect model.

**Understanding `fig` and `axes`:**
- `plt.subplots(1, 2, ...)` creates one row with two plotting areas.
- `fig` represents the complete figure.
- `axes` stores the individual plotting areas.
- `axes[0]` means the first (left) plot.
- `axes[1]` means the second (right) plot.
- `ax=axes[0]` tells `plot_acf()` to draw inside the left plot area.
- `ax=axes[1]` tells `plot_pacf()` to draw inside the right plot area.

`lags=36` displays up to 36 months of lag relationships. `method="ywm"` selects the calculation method used for PACF.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(first_difference, lags=36, ax=axes[0])
axes[0].set_title("ACF of first-differenced log series")

plot_pacf(first_difference, lags=36, method="ywm", ax=axes[1])
axes[1].set_title("PACF of first-differenced log series")

plt.tight_layout()
plt.show()

## 14. Split the data into training and testing sets

A forecasting model must be tested on observations that occur **after** the observations used for training.

We use the first 80% of the timeline for training and the final 20% for testing. The code calculates the exact split from the number of observations.

### What do training and testing mean?

- **Training data:** historical observations the model is allowed to learn from.
- **Testing data:** later observations hidden from the model during training. We use them to check how well the model forecasts unseen months.

### Why split chronologically?

Imagine predicting December using a model that has already seen January of the following year. That would leak future information into the past. A chronological split avoids this problem.

In the code:
- `split_point` is the row position where we divide the series.
- `.iloc[:split_point]` selects rows from the beginning up to, but not including, the split point.
- `.iloc[split_point:]` selects the split point and all rows after it.

We keep the time order intact and do not shuffle the rows.

In [ ]:
split_point = int(len(log_passengers) * 0.80)

train = log_passengers.iloc[:split_point]
test = log_passengers.iloc[split_point:]

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training ends:", train.index[-1])
print("Testing starts:", test.index[0])

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train, label="Training data")
plt.plot(test, label="Testing data")
plt.title("Chronological train-test split")
plt.xlabel("Month")
plt.ylabel("Log passengers")
plt.legend()
plt.grid(True)
plt.show()

## 15. Create a simple baseline forecast

Before using advanced models, build a simple benchmark.

For monthly data with yearly seasonality, a **seasonal naive forecast** predicts that each future month will equal the value from 12 months earlier.

Example: the forecast for January 1960 is the observed value from January 1959.

A complex model should ideally improve on this simple baseline.

In [ ]:
baseline_forecast = log_passengers.shift(12).loc[test.index]

plt.figure(figsize=(12, 5))
plt.plot(test, label="Actual test values")
plt.plot(baseline_forecast, label="Seasonal naive forecast")
plt.title("Seasonal naive baseline")
plt.legend()
plt.grid(True)
plt.show()

## 16. Forecast evaluation metrics

We compare predicted values with actual values.

- **MAE:** average absolute prediction error. Easy to interpret in the same unit as the values.
- **MSE:** average squared error. Large errors receive a larger penalty.
- **RMSE:** square root of MSE. It is in the same unit as the values.

We evaluate on the original passenger-count scale, not the log scale, so the errors are easier to explain.

In [ ]:
def evaluate_forecast(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)

    return pd.Series({
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

actual_test = passengers.loc[test.index]
baseline_original_scale = np.exp(baseline_forecast)

evaluate_forecast(actual_test, baseline_original_scale)

## 17. Autoregression (AR)

An **Autoregressive model** predicts the current value using previous values of the same series.

For example, an AR model may use the previous month's passenger level to estimate this month's level.

The parameter `lags` tells the model which previous observations it can use. Here, we use lags 1 through 12 as a simple demonstration.

We fit the model to the training data only.

In [ ]:
ar_model = AutoReg(train, lags=12, trend="c")
ar_results = ar_model.fit()

print(ar_results.summary())

### Forecast with the AR model

The model forecasts 24 months, matching the length of our test set. Because the model was trained on log passenger counts, we use `np.exp()` to convert predictions back to passenger counts.

In [ ]:
ar_forecast_log = ar_results.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    dynamic=False
)

ar_forecast = pd.Series(
    np.exp(np.asarray(ar_forecast_log)),
    index=test.index
)

evaluate_forecast(actual_test, ar_forecast)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual_test, label="Actual")
plt.plot(ar_forecast, label="AR forecast")
plt.title("AR: actual vs forecast")
plt.legend()
plt.grid(True)
plt.show()

## 18. ARIMA model

ARIMA stands for:

- **AR – Autoregression:** uses past values.
- **I – Integrated:** uses differencing to handle changes in level.
- **MA – Moving Average:** uses past forecast errors.

ARIMA is written as:

\[
ARIMA(p,d,q)
\]

- `p`: number of autoregressive terms.
- `d`: number of ordinary differences.
- `q`: number of moving-average terms.

For this section, we model the log-transformed series. The ARIMA model performs the required differencing internally according to `d`; therefore, we pass the undifferenced training series to it. Do not manually difference the data and then also ask ARIMA to difference it again.

### Fit a first ARIMA model

We start with a small example order. These are teaching values, not guaranteed optimal parameters.

In [ ]:
arima_model = ARIMA(train, order=(1, 1, 1))
arima_results = arima_model.fit()

print(arima_results.summary())
print("AIC:", arima_results.aic)

### What is AIC?

AIC means **Akaike Information Criterion**. It balances how well a model fits the training data against how many parameters it uses.

When comparing models fitted to the same data using comparable likelihoods:
- Lower AIC is preferred.
- AIC is a relative comparison, not an accuracy percentage.
- A lower AIC does not guarantee better forecasts on unseen data.

Formula:

\[
AIC = 2k - 2\ln(L)
\]

Here, `k` is the number of estimated parameters and `L` is the maximum likelihood.

### Forecast and evaluate ARIMA

The forecast is produced on the log scale, then converted back to passenger counts.

In [ ]:
arima_forecast_log = arima_results.forecast(steps=len(test))
arima_forecast = pd.Series(
    np.exp(np.asarray(arima_forecast_log)),
    index=test.index
)

evaluate_forecast(actual_test, arima_forecast)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual_test, label="Actual")
plt.plot(arima_forecast, label="ARIMA forecast")
plt.title("ARIMA: actual vs forecast")
plt.legend()
plt.grid(True)
plt.show()

## 19. Compare a small set of ARIMA orders using AIC

Rather than guessing one order, we can try a small number of combinations.

For a beginner-friendly search:
- `p` ranges from 0 to 2.
- `d` ranges from 0 to 1.
- `q` ranges from 0 to 2.

This gives 18 combinations. We fit each model on the training data and record its AIC. Some combinations may fail to converge, so we skip failed fits and keep going.

This is a small demonstration search, not an exhaustive search of every possible ARIMA model.

In [ ]:
import itertools

p_values = range(0, 3)
d_values = range(0, 2)
q_values = range(0, 3)

arima_orders = list(itertools.product(p_values, d_values, q_values))
print("Number of ARIMA combinations:", len(arima_orders))

In [ ]:
arima_candidates = []

for order in arima_orders:
    try:
        fitted_model = ARIMA(train, order=order).fit()
        arima_candidates.append({
            "order": order,
            "AIC": fitted_model.aic
        })
    except Exception:
        pass

arima_results_table = pd.DataFrame(arima_candidates)
arima_results_table = arima_results_table.sort_values("AIC")

arima_results_table.head(10)

### Select the lowest-AIC ARIMA model

We select the order with the smallest AIC from the models that fitted successfully. We then fit that selected model and evaluate its forecast on the untouched test period.

In [ ]:
best_arima_order = arima_results_table.iloc[0]["order"]
print("Best ARIMA order:", best_arima_order)

best_arima_model = ARIMA(train, order=best_arima_order).fit()
best_arima_forecast_log = best_arima_model.forecast(steps=len(test))

best_arima_forecast = pd.Series(
    np.exp(np.asarray(best_arima_forecast_log)),
    index=test.index
)

evaluate_forecast(actual_test, best_arima_forecast)

## 20. SARIMA: ARIMA with seasonality

The AirPassengers series has a repeating yearly pattern. SARIMA extends ARIMA by modelling seasonal behaviour explicitly.

SARIMA is written as:

\[
SARIMA(p,d,q)(P,D,Q,s)
\]

Non-seasonal terms:
- `p`: autoregressive order
- `d`: ordinary differencing order
- `q`: moving-average order

Seasonal terms:
- `P`: seasonal autoregressive order
- `D`: seasonal differencing order
- `Q`: seasonal moving-average order
- `s`: number of observations in one seasonal cycle

For monthly data with yearly seasonality, `s = 12`.

SARIMA can model both short-term relationships and repeating seasonal patterns.

### Fit a SARIMA model

We begin with a reasonable example configuration. The model is fitted on the training portion of the log-transformed series.

`SARIMAX` is the statsmodels implementation that supports seasonal ARIMA and optional external predictors. Here we use it as a SARIMA model without external predictors.

In [ ]:
sarima_order = (1, 1, 1)
seasonal_order = (1, 1, 1, 12)

sarima_model = SARIMAX(
    train,
    order=sarima_order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_results = sarima_model.fit(disp=False)

print("AIC:", sarima_results.aic)

### Forecast the test period with SARIMA

We forecast the same 24 months as the test set. We also obtain a confidence interval. Since the model was fitted on log values, we exponentiate the forecast and interval bounds to return to the passenger-count scale.

In [ ]:
sarima_prediction = sarima_results.get_forecast(steps=len(test))

sarima_forecast = pd.Series(
    np.exp(np.asarray(sarima_prediction.predicted_mean)),
    index=test.index
)

sarima_interval_log = sarima_prediction.conf_int()
sarima_lower = pd.Series(
    np.exp(np.asarray(sarima_interval_log.iloc[:, 0])),
    index=test.index
)
sarima_upper = pd.Series(
    np.exp(np.asarray(sarima_interval_log.iloc[:, 1])),
    index=test.index
)

evaluate_forecast(actual_test, sarima_forecast)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual_test, label="Actual")
plt.plot(sarima_forecast, label="SARIMA forecast")
plt.fill_between(
    test.index,
    sarima_lower,
    sarima_upper,
    alpha=0.2,
    label="Approx. 95% forecast interval"
)
plt.title("SARIMA: actual vs forecast")
plt.xlabel("Month")
plt.ylabel("Passengers")
plt.legend()
plt.grid(True)
plt.show()

## 21. Compare all models fairly

All models were evaluated on the same 24-month test period and on the original passenger-count scale.

A lower MAE or RMSE means smaller forecast errors on this test set. The best model is not chosen from AIC alone: test-set accuracy and forecast diagnostics also matter.

In [ ]:
comparison = pd.DataFrame({
    "Seasonal Naive": evaluate_forecast(actual_test, baseline_original_scale),
    "AR": evaluate_forecast(actual_test, ar_forecast),
    "ARIMA": evaluate_forecast(actual_test, arima_forecast),
    "Best AIC ARIMA": evaluate_forecast(actual_test, best_arima_forecast),
    "SARIMA": evaluate_forecast(actual_test, sarima_forecast)
}).T

comparison.sort_values("RMSE")

### Important teaching point

The lowest test RMSE is the best result **among these models on this particular test period**. It does not prove that the model will always be best on future data. Different test periods can produce different rankings.

A good forecasting workflow also checks residuals, compares against a baseline, and avoids using test data while selecting model settings.

## 22. Forecast the next 24 months

After choosing a model configuration, we can fit it again using all available historical observations. This allows the final model to learn from the complete dataset.

We then forecast the next 24 months. These future values are predictions, not actual observed passenger counts.

In [ ]:
final_model = SARIMAX(
    log_passengers,
    order=sarima_order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_results = final_model.fit(disp=False)
future_prediction = final_results.get_forecast(steps=24)

### Convert the forecast back to passenger counts

The model predicts on the log scale. `np.exp()` reverses the log transformation.

We create 24 future month-start dates beginning in January 1961.

In [ ]:
future_dates = pd.date_range(
    start=passengers.index[-1] + pd.offsets.MonthBegin(1),
    periods=24,
    freq="MS"
)

future_forecast = pd.Series(
    np.exp(np.asarray(future_prediction.predicted_mean)),
    index=future_dates,
    name="Forecast"
)

future_interval_log = future_prediction.conf_int()

future_lower = pd.Series(
    np.exp(np.asarray(future_interval_log.iloc[:, 0])),
    index=future_dates
)
future_upper = pd.Series(
    np.exp(np.asarray(future_interval_log.iloc[:, 1])),
    index=future_dates
)

future_forecast.head()

### Plot historical values and future predictions

The historical line shows observed passenger counts. The forecast line shows the model's predictions for the next two years. The shaded area represents the model's approximate 95% forecast interval.

In [ ]:
plt.figure(figsize=(13, 6))

plt.plot(passengers, label="Historical passengers")
plt.plot(future_forecast, label="Next 24 months forecast")
plt.fill_between(
    future_dates,
    future_lower,
    future_upper,
    alpha=0.2,
    label="Approx. 95% forecast interval"
)

plt.title("AirPassengers: 24-month SARIMA forecast")
plt.xlabel("Month")
plt.ylabel("Number of passengers")
plt.legend()
plt.grid(True)
plt.show()

## 23. What is SARIMAX and what is an exogenous variable?

SARIMAX extends SARIMA by allowing **exogenous variables** (also called external predictors).

An exogenous variable is information outside the target series that may help explain or predict it.

Examples:
- Advertising spend when forecasting sales
- Temperature when forecasting electricity demand
- Public holidays when forecasting store traffic

The key requirement is that future values of the exogenous variable must be available when we make a future forecast.

### Important note about this dataset

The number of days in a month is known in advance, so it can technically be supplied as an exogenous variable. However, it may not add meaningful predictive information for airline passenger totals. Including a variable just because it is available does not guarantee a better model.

The following is an optional demonstration of the SARIMAX interface, not a claim that month length improves this forecast.

In [ ]:
# Historical external variable: number of days in each month
exog_train = pd.DataFrame(
    {"DaysInMonth": passengers.index.days_in_month},
    index=passengers.index
)

# Future external variable: values must be known for every forecast month
exog_future = pd.DataFrame(
    {"DaysInMonth": future_dates.days_in_month},
    index=future_dates
)

print(exog_train.head())
print(exog_future.head())

In [ ]:
sarimax_with_exog = SARIMAX(
    log_passengers,
    exog=exog_train,
    order=sarima_order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_exog_results = sarimax_with_exog.fit(disp=False)

exog_forecast_result = sarimax_exog_results.get_forecast(
    steps=24,
    exog=exog_future
)

exog_forecast = pd.Series(
    np.exp(np.asarray(exog_forecast_result.predicted_mean)),
    index=future_dates
)

exog_forecast.head()

## 24. Final recap

| Concept | Simple meaning |
|---|---|
| Time series | Observations recorded in time order |
| Trend | Long-term upward or downward movement |
| Seasonality | Pattern that repeats at a fixed interval |
| Stationarity | Statistical behaviour remains broadly stable over time |
| ACF | Relationship between values and their lagged values |
| PACF | Direct lag relationship after accounting for shorter lags |
| Log transformation | Compresses large values and can help stabilize variation |
| Differencing | Subtracts an earlier value to focus on change |
| AR | Uses past values to predict the current value |
| ARIMA | Combines AR, differencing, and moving-average terms |
| SARIMA | ARIMA with seasonal terms |
| SARIMAX | SARIMA with external predictors |
| MAE / RMSE | Measures of forecast error |
| AIC | Relative model comparison criterion |

### The complete workflow

1. Load and inspect the data.
2. Convert the date column and set a time index.
3. Plot the series and identify trend and seasonality.
4. Investigate stationarity using plots and the ADF test.
5. Apply transformations or differencing when appropriate.
6. Split the data chronologically.
7. Build a simple baseline.
8. Fit forecasting models using training data.
9. Evaluate predictions on the test period.
10. Refit the selected model on all historical data and forecast the future.

### Questions to ask students

1. Why should we not randomly shuffle time series observations?
2. What is the difference between trend and seasonality?
3. What does lag 12 mean for monthly data?
4. What are the null and alternative hypotheses of the ADF test?
5. How is differencing different from log transformation?
6. What do `p`, `d`, and `q` represent in ARIMA?
7. Why does SARIMA include a seasonal period?
8. Why do we compare a model with a simple baseline?
9. What is the difference between AIC and RMSE?
10. Why must future exogenous values be supplied to SARIMAX?

**End of notebook — Time Series Analysis and Forecasting with AirPassengers.**